# Day 3: Manifolds, Point Clouds, and Equivariance

**Geometric Deep Learning Workshop**

---

## Learning Objectives

1. Generate and visualize 3D point clouds
2. Understand and implement key ideas from PointNet (symmetric functions)
3. Explore SE(3) equivariance concepts for geometric data
4. Combine topological features (from TDA) with GNN node features

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch: {torch.__version__}")

---

## 1. Point Cloud Generation and Visualization

A **point cloud** is a set of points $\{p_1, p_2, \dots, p_N\} \subset \mathbb{R}^3$.
Unlike graphs, point clouds have no explicit connectivity; the structure must be
inferred from spatial proximity.

Key challenges:
- **Unordered**: no canonical ordering of points
- **Irregular**: non-uniform spacing
- **Transformation invariance**: rotations and translations should not change semantics

In [ ]:
def generate_sphere(n_points=500, radius=1.0, noise=0.02):
    """Generate points on a sphere with optional noise."""
    theta = np.random.uniform(0, 2 * np.pi, n_points)
    phi = np.arccos(np.random.uniform(-1, 1, n_points))
    x = radius * np.sin(phi) * np.cos(theta)
    y = radius * np.sin(phi) * np.sin(theta)
    z = radius * np.cos(phi)
    points = np.stack([x, y, z], axis=1)
    points += np.random.randn(*points.shape) * noise
    return points


def generate_torus(n_points=500, R=1.0, r=0.4, noise=0.02):
    """Generate points on a torus with major radius R and minor radius r."""
    theta = np.random.uniform(0, 2 * np.pi, n_points)
    phi = np.random.uniform(0, 2 * np.pi, n_points)
    x = (R + r * np.cos(phi)) * np.cos(theta)
    y = (R + r * np.cos(phi)) * np.sin(theta)
    z = r * np.sin(phi)
    points = np.stack([x, y, z], axis=1)
    points += np.random.randn(*points.shape) * noise
    return points


def generate_bunny_approx(n_points=500, noise=0.02):
    """Generate a rough bunny-like shape from ellipsoids."""
    # Body: ellipsoid
    n_body = int(n_points * 0.6)
    body = np.random.randn(n_body, 3) * [0.3, 0.25, 0.4]
    # Head: sphere on top
    n_head = int(n_points * 0.25)
    head = np.random.randn(n_head, 3) * 0.2 + [0, 0, 0.5]
    # Ears: two elongated shapes
    n_ear = n_points - n_body - n_head
    n_each = n_ear // 2
    ear_l = np.random.randn(n_each, 3) * [0.04, 0.04, 0.2] + [-0.1, 0, 0.8]
    ear_r = np.random.randn(n_ear - n_each, 3) * [0.04, 0.04, 0.2] + [0.1, 0, 0.8]
    points = np.vstack([body, head, ear_l, ear_r])
    points += np.random.randn(*points.shape) * noise
    return points


# Generate point clouds
sphere = generate_sphere(500)
torus = generate_torus(500)
bunny = generate_bunny_approx(500)

print(f"Sphere: {sphere.shape}")
print(f"Torus:  {torus.shape}")
print(f"Bunny:  {bunny.shape}")

In [ ]:
# Visualize the point clouds
fig = plt.figure(figsize=(16, 5))

datasets = [('Sphere', sphere), ('Torus', torus), ('Bunny (approx)', bunny)]

for idx, (name, points) in enumerate(datasets):
    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
    ax.scatter(points[:, 0], points[:, 1], points[:, 2],
              s=5, alpha=0.6, c=points[:, 2], cmap='viridis')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(name, fontsize=13)
    # Equal aspect ratio
    max_range = np.ptp(points, axis=0).max() / 2
    mid = points.mean(axis=0)
    ax.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax.set_ylim(mid[1] - max_range, mid[1] + max_range)
    ax.set_zlim(mid[2] - max_range, mid[2] + max_range)

plt.tight_layout()
plt.show()

In [ ]:
# Constructing graphs from point clouds via k-nearest neighbors
from scipy.spatial import KDTree

def knn_graph(points, k=6):
    """Build a k-nearest neighbor graph from a point cloud."""
    tree = KDTree(points)
    edges = []
    for i in range(len(points)):
        # Query k+1 because the point itself is included
        dists, indices = tree.query(points[i], k=k+1)
        for j in indices[1:]:  # skip self
            edges.append((i, j))
    return edges


# Build kNN graph for the torus
k = 8
torus_edges = knn_graph(torus, k=k)
print(f"Torus kNN graph (k={k}): {len(torus)} nodes, {len(torus_edges)} directed edges")

# Visualize the graph structure on a subset
subset = 80  # use fewer points for clarity
torus_small = torus[:subset]
edges_small = knn_graph(torus_small, k=4)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Draw edges
for i, j in edges_small:
    ax.plot([torus_small[i, 0], torus_small[j, 0]],
            [torus_small[i, 1], torus_small[j, 1]],
            [torus_small[i, 2], torus_small[j, 2]],
            color='gray', alpha=0.2, linewidth=0.5)

# Draw nodes
ax.scatter(torus_small[:, 0], torus_small[:, 1], torus_small[:, 2],
          s=30, c=torus_small[:, 2], cmap='viridis', edgecolors='black',
          linewidths=0.3, zorder=5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title(f'kNN Graph on Torus (k=4, {subset} points)', fontsize=13)
plt.tight_layout()
plt.show()

---

## 2. PointNet Architecture

**PointNet** (Qi et al., 2017) processes point clouds directly. Key insight:
use **symmetric functions** (like max-pooling) to achieve permutation invariance.

Architecture:
1. **Per-point MLP**: shared MLP applied independently to each point
2. **Symmetric aggregation**: max-pool across all points to get a global descriptor
3. **Classification/segmentation head**: MLP on the global feature

$$f(\{x_1, \dots, x_N\}) = \gamma\left(\max_{i=1}^N h(x_i)\right)$$

where $h$ is a per-point MLP, $\max$ is element-wise max-pool, and $\gamma$ is the classifier.

In [ ]:
# Demonstrate why we need symmetric functions

# A point cloud is a SET - order should not matter
points_ordered = torch.tensor([[1.0, 2.0, 3.0],
                                [4.0, 5.0, 6.0],
                                [7.0, 8.0, 9.0]])

# Shuffle the points
perm = torch.randperm(3)
points_shuffled = points_ordered[perm]

print("Original order:")
print(points_ordered)
print(f"\nShuffled (permutation {perm.tolist()}):")
print(points_shuffled)

# Non-symmetric function (flatten + linear): NOT permutation invariant
flat_orig = points_ordered.flatten()
flat_shuf = points_shuffled.flatten()
print(f"\nFlattened original:  {flat_orig.tolist()}")
print(f"Flattened shuffled:  {flat_shuf.tolist()}")
print(f"Same? {torch.allclose(flat_orig, flat_shuf)}")

# Symmetric functions: PERMUTATION INVARIANT
print(f"\nMax-pool original:   {points_ordered.max(dim=0).values.tolist()}")
print(f"Max-pool shuffled:   {points_shuffled.max(dim=0).values.tolist()}")
print(f"Same? {torch.allclose(points_ordered.max(dim=0).values, points_shuffled.max(dim=0).values)}")

print(f"\nSum original:        {points_ordered.sum(dim=0).tolist()}")
print(f"Sum shuffled:        {points_shuffled.sum(dim=0).tolist()}")
print(f"Same? {torch.allclose(points_ordered.sum(dim=0), points_shuffled.sum(dim=0))}")

In [ ]:
class TNet(nn.Module):
    """Spatial Transformer Network for input alignment.

    Learns a transformation matrix to canonicalize the input.
    For 3D points, it predicts a 3x3 rotation-like matrix.
    """

    def __init__(self, k=3):
        super().__init__()
        self.k = k

        self.mlp1 = nn.Sequential(
            nn.Linear(k, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(),
        )

        self.fc = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, k * k),
        )

    def forward(self, x):
        """x: [batch_size, n_points, k]"""
        batch_size = x.size(0)
        n_points = x.size(1)

        # Per-point features
        x_flat = x.reshape(-1, self.k)  # [B*N, k]
        h = self.mlp1(x_flat)           # [B*N, 256]
        h = h.reshape(batch_size, n_points, -1)  # [B, N, 256]

        # Symmetric aggregation (max-pool over points)
        h = h.max(dim=1).values  # [B, 256]

        # Predict transformation matrix
        transform = self.fc(h)  # [B, k*k]
        transform = transform.reshape(batch_size, self.k, self.k)

        # Initialize as identity
        identity = torch.eye(self.k, device=x.device).unsqueeze(0)
        transform = transform + identity

        return transform


class PointNet(nn.Module):
    """PointNet for point cloud classification.

    Architecture:
        Input (Nx3) -> T-Net -> MLP(64,128,1024) -> MaxPool -> MLP(512,256,C)
    """

    def __init__(self, num_classes=10, num_points=1024):
        super().__init__()
        self.num_points = num_points

        # Input transform (3x3)
        self.input_transform = TNet(k=3)

        # Shared MLP (per-point feature extraction)
        self.mlp1 = nn.Sequential(
            nn.Linear(3, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 1024), nn.BatchNorm1d(1024), nn.ReLU(),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        """
        Args:
            x: Point cloud [batch_size, n_points, 3]

        Returns:
            Class logits [batch_size, num_classes]
        """
        batch_size = x.size(0)
        n_points = x.size(1)

        # Step 1: Input alignment via T-Net
        transform = self.input_transform(x)  # [B, 3, 3]
        x = torch.bmm(x, transform)          # [B, N, 3]

        # Step 2: Per-point MLP
        x = x.reshape(-1, 3)        # [B*N, 3]
        x = self.mlp1(x)            # [B*N, 1024]
        x = x.reshape(batch_size, n_points, -1)  # [B, N, 1024]

        # Step 3: Symmetric aggregation (max-pool)
        global_feat = x.max(dim=1).values  # [B, 1024]

        # Step 4: Classification
        logits = self.classifier(global_feat)  # [B, num_classes]
        return logits


# Create model
pointnet = PointNet(num_classes=3, num_points=500)
print(pointnet)
print(f"\nTotal parameters: {sum(p.numel() for p in pointnet.parameters()):,}")

In [ ]:
# Train PointNet on our synthetic point clouds

def generate_dataset(n_samples_per_class=100, n_points=500):
    """Generate a dataset of sphere, torus, and bunny point clouds."""
    data = []
    labels = []
    generators = [generate_sphere, generate_torus, generate_bunny_approx]

    for class_idx, gen_fn in enumerate(generators):
        for _ in range(n_samples_per_class):
            # Add random rotation for data augmentation
            angle = np.random.uniform(0, 2 * np.pi)
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            R = np.array([[cos_a, -sin_a, 0],
                          [sin_a, cos_a, 0],
                          [0, 0, 1]])
            pts = gen_fn(n_points, noise=0.03) @ R.T
            data.append(pts)
            labels.append(class_idx)

    data = np.array(data, dtype=np.float32)
    labels = np.array(labels, dtype=np.int64)

    # Shuffle
    perm = np.random.permutation(len(data))
    return data[perm], labels[perm]


# Generate dataset
X_all, y_all = generate_dataset(n_samples_per_class=80, n_points=256)
n_train = int(0.8 * len(X_all))

X_train, y_train = X_all[:n_train], y_all[:n_train]
X_test, y_test = X_all[n_train:], y_all[n_train:]

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Point cloud shape: {X_train[0].shape}")
print(f"Class distribution (train): {np.bincount(y_train)}")

In [ ]:
# Use a smaller PointNet for faster training
class PointNetSmall(nn.Module):
    """Lightweight PointNet for quick demonstration."""

    def __init__(self, num_classes=3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(3, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        """x: [B, N, 3]"""
        B, N, _ = x.shape
        # Per-point features
        h = self.mlp(x.reshape(-1, 3))  # [B*N, 256]
        h = h.reshape(B, N, -1)          # [B, N, 256]
        # Max-pool (symmetric aggregation)
        global_feat = h.max(dim=1).values  # [B, 256]
        # Classify
        return self.classifier(global_feat)


model_pn = PointNetSmall(num_classes=3)
optimizer = torch.optim.Adam(model_pn.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Convert to tensors
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)

# Training loop
batch_size = 32
train_losses = []
test_accs = []

for epoch in range(50):
    model_pn.train()
    epoch_loss = 0
    n_batches = 0

    # Mini-batch training
    perm = torch.randperm(len(X_train_t))
    for i in range(0, len(X_train_t), batch_size):
        idx = perm[i:i+batch_size]
        x_batch = X_train_t[idx]
        y_batch = y_train_t[idx]

        optimizer.zero_grad()
        out = model_pn(x_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    # Evaluate
    model_pn.eval()
    with torch.no_grad():
        test_out = model_pn(X_test_t)
        test_pred = test_out.argmax(dim=1)
        test_acc = (test_pred == y_test_t).float().mean().item()

    train_losses.append(epoch_loss / n_batches)
    test_accs.append(test_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:2d} | Loss: {train_losses[-1]:.4f} | Test Acc: {test_acc:.4f}")

print(f"\nBest test accuracy: {max(test_accs):.4f}")

In [ ]:
# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, color='steelblue')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(test_accs, color='coral')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Test Accuracy (Shape Classification)')
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Verify permutation invariance
model_pn.eval()
sample = X_test_t[0:1]  # [1, N, 3]

with torch.no_grad():
    out_original = model_pn(sample)

    # Randomly permute the points
    perm = torch.randperm(sample.size(1))
    sample_permuted = sample[:, perm, :]
    out_permuted = model_pn(sample_permuted)

print("Output with original order: ", out_original.numpy().round(4))
print("Output with permuted order: ", out_permuted.numpy().round(4))
print(f"Outputs are equal: {torch.allclose(out_original, out_permuted, atol=1e-5)}")
print("\nPointNet is permutation invariant because of the max-pool aggregation!")

---

## 3. Equivariant Neural Networks

### SE(3) Equivariance

The group **SE(3)** consists of 3D rotations and translations.
A function $f$ is **SE(3)-equivariant** if:

$$f(T \cdot x) = T \cdot f(x)$$

for any transformation $T \in SE(3)$.

- **Invariance**: $f(T \cdot x) = f(x)$ (output does not change)
- **Equivariance**: $f(T \cdot x) = T \cdot f(x)$ (output transforms consistently)

PointNet achieves **invariance** through max-pooling. But for tasks like
per-point prediction (segmentation), we need **equivariance**.

In [ ]:
# Demonstrate the difference between invariance and equivariance

def rotation_matrix_z(angle):
    """Rotation matrix around the Z-axis."""
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]], dtype=np.float32)


def rotation_matrix_3d(alpha, beta, gamma):
    """General 3D rotation matrix (Euler angles: ZYX convention)."""
    Rz = rotation_matrix_z(alpha)
    cy, sy = np.cos(beta), np.sin(beta)
    Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]], dtype=np.float32)
    cx, sx = np.cos(gamma), np.sin(gamma)
    Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]], dtype=np.float32)
    return Rz @ Ry @ Rx


# Original point cloud
points = generate_sphere(200, noise=0.01)

# Apply rotation
R = rotation_matrix_3d(np.pi/4, np.pi/6, np.pi/3)
points_rotated = points @ R.T

# Invariant feature: centroid distance distribution
centroid_orig = points.mean(axis=0)
dists_orig = np.linalg.norm(points - centroid_orig, axis=1)

centroid_rot = points_rotated.mean(axis=0)
dists_rot = np.linalg.norm(points_rotated - centroid_rot, axis=1)

# Equivariant feature: per-point normals (estimated from local PCA)
from scipy.spatial import KDTree

def estimate_normals(pts, k=10):
    """Estimate surface normals using local PCA."""
    tree = KDTree(pts)
    normals = np.zeros_like(pts)
    for i in range(len(pts)):
        _, idx = tree.query(pts[i], k=k)
        neighbors = pts[idx]
        centered = neighbors - neighbors.mean(axis=0)
        cov = centered.T @ centered
        eigvals, eigvecs = np.linalg.eigh(cov)
        normals[i] = eigvecs[:, 0]  # smallest eigenvector = normal
    return normals

normals_orig = estimate_normals(points)
normals_rot = estimate_normals(points_rotated)

# Check: rotated normals should equal R @ original normals
normals_orig_rotated = normals_orig @ R.T

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Invariant check
axes[0].hist(dists_orig, bins=30, alpha=0.6, label='Original', color='steelblue')
axes[0].hist(dists_rot, bins=30, alpha=0.6, label='Rotated', color='coral')
axes[0].set_xlabel('Distance from centroid')
axes[0].set_ylabel('Count')
axes[0].set_title('Invariant Feature: Distance Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Equivariant check
# Compare R @ normals_orig vs normals_rotated (should be similar up to sign)
cos_sim = np.abs((normals_orig_rotated * normals_rot).sum(axis=1))
axes[1].hist(cos_sim, bins=30, color='seagreen', alpha=0.7)
axes[1].set_xlabel('|cos(angle)| between R*n_orig and n_rot')
axes[1].set_ylabel('Count')
axes[1].set_title('Equivariant Check: Normal Consistency')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Distance distributions match: {np.allclose(sorted(dists_orig), sorted(dists_rot), atol=0.01)}")
print(f"Mean normal alignment: {cos_sim.mean():.4f} (1.0 = perfectly equivariant)")

In [ ]:
# Simple equivariant layer: uses relative positions (inherently SE(3)-invariant edges)

class SE3InvariantLayer(nn.Module):
    """A simple layer that operates on pairwise distances (SE(3)-invariant).

    By using ||x_i - x_j|| as edge features instead of absolute positions,
    we achieve invariance to rotations and translations.
    """

    def __init__(self, in_features, out_features, k_neighbors=8):
        super().__init__()
        self.k = k_neighbors
        # Message MLP: operates on [node_feat_i, node_feat_j, distance]
        self.message_mlp = nn.Sequential(
            nn.Linear(2 * in_features + 1, out_features),
            nn.ReLU(),
            nn.Linear(out_features, out_features),
        )

    def forward(self, positions, features):
        """
        Args:
            positions: [B, N, 3] - 3D coordinates
            features:  [B, N, F] - node features

        Returns:
            updated_features: [B, N, out_features]
        """
        B, N, _ = positions.shape

        # Compute pairwise distances
        # diff[b,i,j] = positions[b,i] - positions[b,j]
        diff = positions.unsqueeze(2) - positions.unsqueeze(1)  # [B, N, N, 3]
        dists = diff.norm(dim=-1)  # [B, N, N]

        # Find k nearest neighbors
        _, knn_idx = dists.topk(self.k + 1, dim=-1, largest=False)  # [B, N, k+1]
        knn_idx = knn_idx[:, :, 1:]  # Remove self [B, N, k]

        # Gather neighbor features and distances
        knn_idx_expanded = knn_idx.unsqueeze(-1).expand(-1, -1, -1, features.size(-1))
        neighbor_feats = features.unsqueeze(1).expand(-1, N, -1, -1)
        neighbor_feats = neighbor_feats.gather(2, knn_idx_expanded)  # [B, N, k, F]

        # Gather distances to neighbors
        neighbor_dists = dists.gather(2, knn_idx)  # [B, N, k]

        # Build messages: [self_feat, neighbor_feat, distance]
        self_feats = features.unsqueeze(2).expand(-1, -1, self.k, -1)  # [B, N, k, F]
        messages = torch.cat([
            self_feats,
            neighbor_feats,
            neighbor_dists.unsqueeze(-1),
        ], dim=-1)  # [B, N, k, 2F+1]

        # Apply message MLP
        messages = self.message_mlp(messages)  # [B, N, k, out]

        # Aggregate (mean)
        updated = messages.mean(dim=2)  # [B, N, out]

        return updated


# Test SE(3) invariance
layer = SE3InvariantLayer(in_features=8, out_features=16, k_neighbors=6)
layer.eval()

# Random point cloud with features
pos = torch.randn(2, 50, 3)
feat = torch.randn(2, 50, 8)

with torch.no_grad():
    out_orig = layer(pos, feat)

# Apply rotation + translation
R_torch = torch.from_numpy(rotation_matrix_3d(0.5, 1.0, 0.3)).unsqueeze(0)  # [1, 3, 3]
t_torch = torch.randn(1, 1, 3)  # translation
pos_transformed = (pos @ R_torch.transpose(1, 2)) + t_torch

with torch.no_grad():
    out_transformed = layer(pos_transformed, feat)

diff = (out_orig - out_transformed).abs().max().item()
print(f"Max difference after SE(3) transformation: {diff:.8f}")
print(f"Outputs are SE(3)-invariant: {diff < 1e-4}")

---

## 4. Combining TDA Features with GNNs

**Topological Data Analysis (TDA)** extracts shape descriptors that are robust to
deformations. We can compute simple topological features and concatenate them
with GNN node features for richer representations.

Key TDA concepts:
- **Persistent homology**: tracks topological features (components, loops, voids) across scales
- **Betti numbers**: $\beta_0$ = connected components, $\beta_1$ = loops, $\beta_2$ = voids
- **Persistence diagrams**: summarize the birth/death of topological features

Here we compute simple graph-topological features to augment node representations.

In [ ]:
# Compute topological node features from a graph

def compute_topo_features(G):
    """Compute topological features for each node in a NetworkX graph.

    Features per node:
    1. Degree
    2. Clustering coefficient
    3. Number of triangles
    4. Closeness centrality
    5. Local efficiency (related to cycle structure)
    6. k-core number (captures nested structure)
    """
    n = G.number_of_nodes()
    features = np.zeros((n, 6), dtype=np.float32)

    # Degree
    for node, deg in G.degree():
        features[node, 0] = deg

    # Clustering coefficient
    clustering = nx.clustering(G)
    for node, cc in clustering.items():
        features[node, 1] = cc

    # Number of triangles
    triangles = nx.triangles(G)
    for node, tri in triangles.items():
        features[node, 2] = tri

    # Closeness centrality
    closeness = nx.closeness_centrality(G)
    for node, cl in closeness.items():
        features[node, 3] = cl

    # k-core number
    core_numbers = nx.core_number(G)
    for node, cn in core_numbers.items():
        features[node, 4] = cn

    # Betweenness centrality (related to bridges/cut vertices)
    betweenness = nx.betweenness_centrality(G)
    for node, bc in betweenness.items():
        features[node, 5] = bc

    return features


# Compute for the Karate Club graph
G_karate = nx.karate_club_graph()
topo_features = compute_topo_features(G_karate)

feature_names = ['Degree', 'Clustering', 'Triangles', 'Closeness', 'k-Core', 'Betweenness']

print("Topological features for Karate Club (first 5 nodes):")
print(f"{'Node':>5}", end='')
for name in feature_names:
    print(f"{name:>12}", end='')
print()
for i in range(5):
    print(f"{i:>5}", end='')
    for j in range(6):
        print(f"{topo_features[i, j]:>12.3f}", end='')
    print()

print(f"\nFeature matrix shape: {topo_features.shape}")

In [ ]:
# Simple persistence-based features using filtration on the graph
# (Approximation without a full TDA library)

def vietoris_rips_betti(points, max_radius=2.0, n_steps=50):
    """Compute approximate Betti-0 curve using Vietoris-Rips-like filtration.

    At each radius threshold, count connected components in the
    proximity graph. This approximates beta_0 from persistent homology.
    """
    from scipy.spatial.distance import pdist, squareform

    dist_matrix = squareform(pdist(points))
    radii = np.linspace(0, max_radius, n_steps)
    betti_0 = []

    for r in radii:
        # Build proximity graph at scale r
        adj = (dist_matrix <= r).astype(int)
        np.fill_diagonal(adj, 0)
        G_r = nx.from_numpy_array(adj)
        n_components = nx.number_connected_components(G_r)
        betti_0.append(n_components)

    return radii, np.array(betti_0)


# Compute Betti-0 curves for our shapes (using small subsets for speed)
n_sample = 100
sphere_sample = generate_sphere(n_sample)
torus_sample = generate_torus(n_sample)

radii_s, betti0_s = vietoris_rips_betti(sphere_sample, max_radius=3.0)
radii_t, betti0_t = vietoris_rips_betti(torus_sample, max_radius=3.0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(radii_s, betti0_s, label='Sphere', color='steelblue', linewidth=2)
ax.plot(radii_t, betti0_t, label='Torus', color='coral', linewidth=2)
ax.set_xlabel('Radius (filtration parameter)', fontsize=12)
ax.set_ylabel('Betti-0 (connected components)', fontsize=12)
ax.set_title('Persistent Betti-0: Sphere vs Torus', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The Betti-0 curve shows how connected components merge as we increase the radius.")
print("Different topologies have distinguishably different curves.")

In [ ]:
# GNN with topological features: concatenate TDA features with learned features
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_networkx

# Load Karate Club
data_kc = KarateClub()[0]

# Compute topological features
G_kc = to_networkx(data_kc, to_undirected=True)
topo_feats = compute_topo_features(G_kc)
topo_tensor = torch.from_numpy(topo_feats)

# Normalize topological features
topo_mean = topo_tensor.mean(dim=0, keepdim=True)
topo_std = topo_tensor.std(dim=0, keepdim=True).clamp(min=1e-6)
topo_normed = (topo_tensor - topo_mean) / topo_std

# Concatenate with original node features
x_augmented = torch.cat([data_kc.x, topo_normed], dim=1)
print(f"Original features shape: {data_kc.x.shape}")
print(f"Topo features shape:     {topo_normed.shape}")
print(f"Augmented features:      {x_augmented.shape}")


class GCN_TDA(nn.Module):
    """GCN with topological feature augmentation."""

    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x


# Compare: baseline GCN vs GCN + TDA features
num_classes = data_kc.y.max().item() + 1


def train_and_eval(model, x, edge_index, y, train_mask, n_epochs=200):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()
    best_acc = 0

    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        out = model(x, edge_index)
        loss = criterion(out[train_mask], y[train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred = model(x, edge_index).argmax(dim=1)
            acc = (pred == y).float().mean().item()
            best_acc = max(best_acc, acc)

    return best_acc


# Run comparison multiple times
baseline_accs = []
tda_accs = []

for trial in range(10):
    torch.manual_seed(trial)

    # Baseline: original features only
    model_base = GCN_TDA(data_kc.num_features, 16, num_classes)
    acc_base = train_and_eval(model_base, data_kc.x, data_kc.edge_index,
                              data_kc.y, data_kc.train_mask)
    baseline_accs.append(acc_base)

    # TDA-augmented
    model_tda = GCN_TDA(x_augmented.size(1), 16, num_classes)
    acc_tda = train_and_eval(model_tda, x_augmented, data_kc.edge_index,
                             data_kc.y, data_kc.train_mask)
    tda_accs.append(acc_tda)

print(f"Baseline GCN:     {np.mean(baseline_accs):.4f} +/- {np.std(baseline_accs):.4f}")
print(f"GCN + TDA feats:  {np.mean(tda_accs):.4f} +/- {np.std(tda_accs):.4f}")

In [ ]:
# Visualize the topological features on the graph
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

pos = nx.spring_layout(G_kc, seed=42)

for idx, (ax, name) in enumerate(zip(axes.flatten(), feature_names)):
    values = topo_features[:, idx]
    sc = nx.draw_networkx_nodes(G_kc, pos, ax=ax, node_color=values,
                                cmap='YlOrRd', node_size=200)
    nx.draw_networkx_edges(G_kc, pos, ax=ax, edge_color='lightgray', width=0.5)
    nx.draw_networkx_labels(G_kc, pos, ax=ax, font_size=7)
    ax.set_title(name, fontsize=12)
    plt.colorbar(sc, ax=ax, shrink=0.8)

plt.suptitle('Topological Node Features on Karate Club', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Summary

| Topic | Key Takeaway |
|-------|-------------|
| Point clouds | Unordered 3D point sets; build graphs via kNN |
| PointNet | Per-point MLP + max-pool = permutation invariant |
| SE(3) equivariance | Use relative positions/distances for invariance; equivariant layers preserve transformation structure |
| TDA + GNNs | Topological features (degree, clustering, persistence) can augment GNN inputs |

**Next**: Day 4 will cover applications: molecular graphs, social networks, and a capstone project.

---

### Exercises

1. **Different aggregations**: Replace max-pool in PointNetSmall with mean-pool and sum-pool. Compare accuracy.
2. **kNN vs radius graphs**: Implement a radius-based graph construction (connect points within distance $r$). Compare with kNN.
3. **Rotation test**: Apply a random rotation to the test set and measure PointNet accuracy. Does it remain invariant? Why or why not?
4. **More TDA features**: Add eigenvector centrality and PageRank to the topological features. Does this help?